In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 1. Configuración de Rutas
base_path = 'data/parquet_v5/'

# 2. Carga Rápida de Bases
def load_and_clean(name, cols=None):
    try:
        df = pd.read_parquet(f'{base_path}{name}.parquet')
        df.columns = df.columns.str.strip().str.lower()
        if 'code' in df.columns:
            df['code'] = df['code'].astype(str).str.replace('.', '', regex=False).str.strip()
            df['code6'] = df['code'].str[:6] # Columna auxiliar para agrupar a nivel subpartida
        return df
    except Exception as e:
        print(f"Error cargando {name}: {e}")
        return pd.DataFrame()

metals = load_and_clean('metals')
tmec = load_and_clean('tmec')
rvc = load_and_clean('rvc')
flip301 = load_and_clean('flip_301')
pharma = load_and_clean('pharma')
auto = load_and_clean('auto')
mhdv = load_and_clean('mhdv')
wood = load_and_clean('wood')
semi = load_and_clean('semi')

escenarios = {}

# --- ESCENARIO 1: Choque de Jerarquías (Auto vs Metales) ---
# Prueba que si un código tiene tanto Auto como Metales, Auto (25%) domine y Metales (junto con Flip 301) sea ignorado.
if not auto.empty and not metals.empty:
    clash_auto_metals = pd.merge(auto[['code6']].drop_duplicates(), metals[['code6']].drop_duplicates(), on='code6')
    escenarios['[Jerarquía] Auto vs Metales (Gana Auto)'] = clash_auto_metals['code6'].head(3).tolist()

# --- ESCENARIO 2: Choque de Jerarquías (Semi vs Auto/MHDV) ---
# Prueba que Semi exente a todos los demás aranceles debajo de él.
if not semi.empty and not auto.empty:
    clash_semi_auto = pd.merge(semi[['code6']].drop_duplicates(), auto[['code6']].drop_duplicates(), on='code6')
    escenarios['[Jerarquía] Semiconductores vs Auto (Gana Semi)'] = clash_semi_auto['code6'].head(3).tolist()

# --- ESCENARIO 3: Metales (Annex I-C) + RVC + TMEC ---
# Prueba el flujo más largo del wizard: Pregunta TMEC -> Pregunta Fundición EUA -> Pregunta RVC -> Slider de Porcentaje.
if not metals.empty and not rvc.empty:
    metals_ic = metals[metals['duty'].astype(str).str.contains('Annex I-C', case=False, na=False)]
    rvc_ic = pd.merge(metals_ic[['code6']], rvc[['code6']], on='code6').drop_duplicates()
    escenarios['[Metales] Annex I-C con RVC (Flujo largo Wizard)'] = rvc_ic['code6'].head(3).tolist()

# --- ESCENARIO 4: Metales (Regla del 15% fuera de Cap. 72, 73, 74, 76) ---
# Prueba que salte la pregunta del peso conjunto del 15% para productos fuera de los capítulos de metales.
if not metals.empty:
    metals_fuera = metals[~metals['code6'].str.startswith(('72', '73', '74', '76'))]
    escenarios['[Metales] Fuera de Capítulos Principales (Regla 15%)'] = metals_fuera['code6'].drop_duplicates().head(3).tolist()

# --- ESCENARIO 5: Autopartes con TMEC ---
# Prueba que se dispare la pregunta de "Knock-down kits" en el wizard si el usuario dice que SÍ es autoparte y SÍ es TMEC.
if not auto.empty and not tmec.empty:
    auto_parts = auto[auto['category'].astype(str).str.lower() == 'autoparts']
    auto_tmec = pd.merge(auto_parts[['code6']], tmec[['code6']], on='code6').drop_duplicates()
    escenarios['[Auto] Autopartes con beneficio TMEC'] = auto_tmec['code6'].head(3).tolist()

# --- ESCENARIO 6: Wood (Kitchen Cabinets) ---
# Prueba que el Wizard detecte "Kitchen Cabinets and Vanities" y pregunte por ellos.
if not wood.empty:
    wood_cabinets = wood[wood['category'].astype(str).str.lower().str.contains('kitchen', na=False)]
    escenarios['[Wood] Kitchen Cabinets (Detona Wizard)'] = wood_cabinets['code6'].drop_duplicates().head(3).tolist()

# --- ESCENARIO 7: Pharma (Annex I) ---
# Prueba la previsualización del 100% y el flujo de preguntas de exención u Onshoring.
if not pharma.empty:
    pharma_annex1 = pharma[pharma['annex'].astype(str).str.upper() == 'ANNEX I']
    escenarios['[Pharma] Annex I (Detona Wizard 100% / Onshoring)'] = pharma_annex1['code6'].drop_duplicates().head(3).tolist()

# --- ESCENARIO 8: Flip 301 (Excepciones Wizard) ---
# Prueba que un código excluido de la 232 pero con scope 'Ex' dispare el wizard de la 301 Labor.
if not flip301.empty:
    flip_ex = flip301[flip301['scope'].astype(str).str.lower().isin(['ex', 'aircraft', 'pharma'])]
    
    # Aseguramos que NO esté en la 232 para que no sea eximido automáticamente
    all_232_codes = pd.concat([
        metals[['code6']], auto[['code6']], mhdv[['code6']], 
        wood[['code6']], semi[['code6']], pharma[['code6']]
    ]).drop_duplicates()
    
    flip_puro = flip_ex[~flip_ex['code6'].isin(all_232_codes['code6'])]
    escenarios['[Flip 301] Pregunta de Scope (Ex/Aircraft/Pharma)'] = flip_puro['code6'].drop_duplicates().head(3).tolist()

# --- ESCENARIO 9: Conflicto a 10 dígitos (Fracciones Mixtas) ---
# Prueba una subpartida que contiene hijos con DIFERENTES aranceles dentro de una misma proclama (ej. un hijo es 25% y otro es 0%).
def find_mixed_10_digits(df, col_eval):
    if df.empty: return []
    # Contamos cuántas tasas/categorías distintas hay por cada código de 6 dígitos
    mixed = df.groupby('code6')[col_eval].nunique().reset_index()
    mixed = mixed[mixed[col_eval] > 1]
    return mixed['code6'].head(3).tolist()

if not metals.empty:
    escenarios['[10 Dígitos Mixtos] Metales (Diferentes tasas en 1 fracción)'] = find_mixed_10_digits(metals, 'duty')

if not pharma.empty:
    escenarios['[10 Dígitos Mixtos] Pharma (Annex I vs Vacío en 1 fracción)'] = find_mixed_10_digits(pharma, 'annex')

# 3. Impresión de Resultados Formateada
print("==================================================================")
print("📊 CÓDIGOS DE PRUEBA PARA DASHBOARD V10 (SECCIÓN 232 & 301 LABOR)")
print("==================================================================\n")

for name, codes in escenarios.items():
    print(f"📌 {name}")
    if codes:
        print(f"   Prueba con: {', '.join(codes)}\n")
    else:
        print("   (No se encontraron códigos para este escenario en tus bases actuales)\n")

📊 CÓDIGOS DE PRUEBA PARA DASHBOARD V10 (SECCIÓN 232 & 301 LABOR)

📌 [Jerarquía] Auto vs Metales (Gana Auto)
   Prueba con: 732010, 732020, 830210

📌 [Jerarquía] Semiconductores vs Auto (Gana Semi)
   (No se encontraron códigos para este escenario en tus bases actuales)

📌 [Metales] Annex I-C con RVC (Flujo largo Wizard)
   Prueba con: 842710, 842720, 842790

📌 [Metales] Fuera de Capítulos Principales (Regla 15%)
   Prueba con: 820239, 820340, 820559

📌 [Auto] Autopartes con beneficio TMEC
   Prueba con: 400912, 400922, 400932

📌 [Wood] Kitchen Cabinets (Detona Wizard)
   Prueba con: 940340, 940360, 940391

📌 [Pharma] Annex I (Detona Wizard 100% / Onshoring)
   Prueba con: 291899, 292219, 292250

📌 [Flip 301] Pregunta de Scope (Ex/Aircraft/Pharma)
   Prueba con: 080590, 081190, 120730

📌 [10 Dígitos Mixtos] Metales (Diferentes tasas en 1 fracción)
   Prueba con: 732510, 732620, 732690

📌 [10 Dígitos Mixtos] Pharma (Annex I vs Vacío en 1 fracción)
   Prueba con: 291899, 292149, 292219

